In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import sklearn
import sys
import os

In [ ]:
print("Python", sys.version)
print("--------")
print("Pandas:", pd.__version__)
print("Numpy:", np.__version__)
print("SKLearn:", sklearn.__version__)

Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
--------
Pandas: 2.2.3
Numpy: 2.1.3
SKLearn: 1.6.1


In [ ]:
DATA_PQT = 'censo2010_pessoas.parquet'
MUNICIPIO = 'Macapá' # Use o que foi especificado para sua equipe
SIGLA_UF = 'AP' # Sigla da UF do município acima

In [ ]:
# Caminho do arquivo
df_censo = pd.read_parquet(DATA_PQT)
df_censo.shape

FileNotFoundError: [Errno 2] No such file or directory: '/data/censo2010_pessoas.parquet'

In [ ]:
# (0,2) 1- Quais colunas possuem campos nulos em df_censo?
# Ordene-as da que tem mais nulos para a que tem menos.

nulos = df_censo.isnull().sum()
nulos_ordenados = nulos[nulos > 0].sort_values(ascending=False)

# Exibe as colunas com valores nulos
nulos_ordenados

1.1 As colunas SUBGRUPO_OCUPACAO, HORAS_TRABALHADAS_SEMANA e RENDA_TRAB_PRINCIPAL apresentam um elevado volume de valores nulos estruturais. Isto acontece porque o questionário do IBGE possui saltos lógicos: as perguntas sobre a ocupação, o horário de trabalho e a remuneração principal não se aplicam a indivíduos que não fazem parte da população economicamente ativa (como crianças, reformados ou desempregados que não procuraram trabalho). Logo, a ausência de dados não é um erro de recolha, mas sim uma indicação válida de "Não Aplicável".

In [ ]:
# (0,25) 2-Investigue a relação entre IDADE e os nulos em RENDA_TOTAL.
# Existe um padrão claro? Baseado nisso, decida um critério de idade mínima...

# Cria uma máscara booleana para identificar os nulos na renda
renda_nula = df_censo['RENDA_TOTAL'].isnull()

# Plota a comparação da distribuição de idades
plt.figure(figsize=(10, 6))
sns.boxplot(x=renda_nula, y=df_censo['IDADE'])
plt.xticks(ticks=[0, 1], labels=['Renda Preenchida', 'Renda Nula'])
plt.title('Distribuição de Idades: Renda Preenchida vs Renda Nula')
plt.ylabel('Idade (anos)')
plt.show()

# Exibe as estatísticas exatas da idade para quem tem renda nula
print("Estatísticas descritivas da IDADE para quem tem RENDA_TOTAL nula:")
print(df_censo[renda_nula]['IDADE'].describe())

In [ ]:
# Aplicar o critério de idade mínima para manter no conjunto de dados (10 anos)
df_censo = df_censo[df_censo['IDADE'] >= 10]

print(f"Novo formato do dataframe após o filtro de idade: {df_censo.shape}")

In [ ]:
# (0,15) 2.1 Depois de aplicar o filtro acima, ainda restam nulos em RENDA_TOTAL?
# Se sim, quantos, e o que isso sugere?

nulos_renda_restantes = df_censo['RENDA_TOTAL'].isnull().sum()
print(f"Nulos restantes em RENDA_TOTAL (idade >= 10): {nulos_renda_restantes}")

In [ ]:
# (0,3) 3-Verifique se há valores de IDADE ou SEXO fora do domínio esperado...
# Trate essas inconsistências.

# Verifica como os dados estão realmente formatados na coluna
print("Valores originais na coluna SEXO:", df_censo['SEXO'].unique())

# Aplica o filtro cobrindo inteiros, strings e floats
filtro_idade_plausivel = df_censo['IDADE'] <= 120
filtro_sexo_valido = df_censo['SEXO'].isin([1, 2, '1', '2', 1.0, 2.0])

df_censo = df_censo[filtro_idade_plausivel & filtro_sexo_valido]

print(f"Formato do dataframe após tratar inconsistências de idade e sexo: {df_censo.shape}")

In [ ]:
# (0,2) 4-Verifique se há linhas completamente duplicadas em df_censo. Remova-as.

duplicadas = df_censo.duplicated().sum()
print(f"Linhas completamente duplicadas encontradas: {duplicadas}")

df_censo = df_censo.drop_duplicates()
print(f"Formato do dataframe após remover duplicadas: {df_censo.shape}")

In [ ]:
# (0,3) 5-Compare RENDA_TOTAL com RENDA_TODOS_TRABALHOS... Identifique casos em que
# RENDA_TOTAL é muito maior... e corrija esses casos.

# Identifica onde a RENDA_TOTAL é mais de 10x maior que a RENDA_TODOS_TRABALHOS
condicao_erro = df_censo['RENDA_TOTAL'] > (10 * df_censo['RENDA_TODOS_TRABALHOS'])
print(f"Casos iniciais com erro de digitação na renda: {condicao_erro.sum()}")

# Corrige o erro deslocando a vírgula repetidamente até o valor ser plausível
while condicao_erro.sum() > 0:
    df_censo.loc[condicao_erro, 'RENDA_TOTAL'] = df_censo.loc[condicao_erro, 'RENDA_TOTAL'] / 10
    condicao_erro = df_censo['RENDA_TOTAL'] > (10 * df_censo['RENDA_TODOS_TRABALHOS'])

print(f"Casos restantes após correção iterativa: {condicao_erro.sum()}")

In [ ]:
# (0,5) 6 Ainda restam nulos em RENDA_TOTAL, COR_RACA, NIVEL_INSTRUCAO e RENDA_DOMICILIAR_PER_CAPITA.
# Decida e aplique uma estratégia de tratamento para cada uma...

# 1. RENDA_TOTAL: Remoção (dropna). Como são poucos casos (pouco mais de mil em milhões),
# removemos as linhas para não introduzir ruído nas estatísticas financeiras com imputações artificiais.
df_censo = df_censo.dropna(subset=['RENDA_TOTAL'])

# 2. COR_RACA e NIVEL_INSTRUCAO: Imputação categórica. Por serem variáveis qualitativas,
# criamos a categoria "Não informado" para preservar a linha e as restantes respostas do indivíduo.
df_censo['COR_RACA'] = df_censo['COR_RACA'].fillna('Não informado')
df_censo['NIVEL_INSTRUCAO'] = df_censo['NIVEL_INSTRUCAO'].fillna('Não informado')

# 3. RENDA_DOMICILIAR_PER_CAPITA: Imputação pela mediana. Sendo uma variável contínua de
# suporte (não é o alvo principal da nossa análise como a RENDA_TOTAL), imputamos a mediana geral.
mediana_domiciliar = df_censo['RENDA_DOMICILIAR_PER_CAPITA'].median()
df_censo['RENDA_DOMICILIAR_PER_CAPITA'] = df_censo['RENDA_DOMICILIAR_PER_CAPITA'].fillna(mediana_domiciliar)

# Verificação final dos nulos nas colunas tratadas
print(df_censo[['RENDA_TOTAL', 'COR_RACA', 'NIVEL_INSTRUCAO', 'RENDA_DOMICILIAR_PER_CAPITA']].isnull().sum())

6 Para tratar os nulos residuais, apliquei estratégias mistas adaptadas à natureza de cada variável. Na RENDA_TOTAL, optei por remover as linhas (dropna), pois o volume de nulos era irrisório face ao tamanho da base, e a imputação poderia distorcer as métricas centrais do estudo. Para COR_RACA e NIVEL_INSTRUCAO (variáveis categóricas), substitui os nulos por "Não informado" para não perder os restantes dados válidos desses indivíduos. Por fim, na RENDA_DOMICILIAR_PER_CAPITA (variável contínua), imputei a mediana, garantindo que a distribuição central não era afetada pelos valores extremos (outliers).

In [ ]:
# Criação da coluna auxiliar GRUPO antes da seção de Análise e Visualização
df_censo['GRUPO'] = np.where(df_censo.NOME_MUNICIPIO == MUNICIPIO, MUNICIPIO, 'Resto do pais')

# Verifica a distribuição entre a vossa cidade e o resto do país
print(df_censo['GRUPO'].value_counts())

In [ ]:
# (0,35) 7-Compare a distribuição de RENDA_TOTAL entre homens e mulheres, tanto
# no seu município quanto no resto do pais. Homens ganham significativamente mais
# que mulheres (confiança de 95%)? Use boxplots e um teste não paramétrico unicaudal.

# 1. Correção para a legenda: Mapear os códigos para nomes explícitos no dataframe
df_censo['SEXO_NOME'] = df_censo['SEXO'].astype(str).replace({
    '1': 'Masculino', '1.0': 'Masculino',
    '2': 'Feminino', '2.0': 'Feminino'
})

# 2. Visualização: Boxplot (sem outliers para facilitar a leitura)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_censo, x='GRUPO', y='RENDA_TOTAL', hue='SEXO_NOME', showfliers=False)
plt.title('Distribuição de Renda Total por Sexo (Sem Outliers)')
plt.xlabel('Grupo')
plt.ylabel('Renda Total (R$)')
plt.legend(title='Sexo')
plt.show()

# 3. Teste Estatístico (Mann-Whitney U)
# Separar os grupos por região
macapa = df_censo[df_censo['GRUPO'] == MUNICIPIO]
resto = df_censo[df_censo['GRUPO'] == 'Resto do pais']

# Separar as rendas por sexo em cada região usando a nova coluna
homens_macapa = macapa[macapa['SEXO_NOME'] == 'Masculino']['RENDA_TOTAL']
mulheres_macapa = macapa[macapa['SEXO_NOME'] == 'Feminino']['RENDA_TOTAL']

homens_resto = resto[resto['SEXO_NOME'] == 'Masculino']['RENDA_TOTAL']
mulheres_resto = resto[resto['SEXO_NOME'] == 'Feminino']['RENDA_TOTAL']

# Aplicar o teste (alternative='greater' testa se o primeiro grupo é estatisticamente maior que o segundo)
stat_m, p_m = stats.mannwhitneyu(homens_macapa, mulheres_macapa, alternative='greater')
stat_r, p_r = stats.mannwhitneyu(homens_resto, mulheres_resto, alternative='greater')

print("\n--- Resultados do Teste Mann-Whitney U (Unicaudal) ---")
print(f"{MUNICIPIO}: Estatística = {stat_m:.2f} | p-valor = {p_m:.2e}")
print(f"Resto do país: Estatística = {stat_r:.2f} | p-valor = {p_r:.2e}")

# Avaliação com 95% de confiança (alpha = 0.05)
alpha = 0.05
print("\nConclusão (Confiança de 95%):")
print(f"{MUNICIPIO}: {'Homens ganham mais (Significativo)' if p_m < alpha else 'Diferença Não Significativa'}")
print(f"Resto do País: {'Homens ganham mais (Significativo)' if p_r < alpha else 'Diferença Não Significativa'}")

7.1 Os resultados do teste de Mann-Whitney U mostram p-valores próximos de zero (muito inferiores ao nível de significância $\alpha = 0.05$) tanto para Macapá quanto para o restante do país. Isso permite-nos rejeitar a hipótese nula com 95% de confiança, comprovando estatisticamente que os homens possuem rendimentos superiores aos das mulheres. A análise dos boxplots corrobora essa conclusão visualmente, demonstrando que a mediana da renda (linha central da caixa) e o terceiro quartil (limite superior da caixa) do sexo masculino estão posicionados acima dos valores correspondentes ao sexo feminino em ambos os recortes geográficos.

In [ ]:
# (0,4) 8-Compare a distribuição de RENDA_TOTAL por cor/raça (COR_RACA), tanto no seu município
# quanto no resto do país. Use um gráfico de violino e um gráfico de barras com intervalo de confiança.
# Qual a melhor visualização e porque?

# 1. Mapeamento do dicionário do IBGE para nomes reais (incluindo o código 9)
dicionario_raca = {
    '1': 'Branca', '1.0': 'Branca',
    '2': 'Preta',  '2.0': 'Preta',
    '3': 'Amarela','3.0': 'Amarela',
    '4': 'Parda',  '4.0': 'Parda',
    '5': 'Indígena','5.0': 'Indígena',
    '9': 'Não informado', '9.0': 'Não informado',
    'Não informado': 'Não informado'
}
df_censo['COR_NOME'] = df_censo['COR_RACA'].astype(str).replace(dicionario_raca)

# 2. Configurar a figura com 2 subplots lado a lado
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Violino
# Limitamos o eixo Y localmente no violino para não ser esmagado pelos outliers extremos
sns.violinplot(data=df_censo, x='GRUPO', y='RENDA_TOTAL', hue='COR_NOME',
               inner='quartile', ax=axes[0])
axes[0].set_title('Violino: Distribuição de Renda por Cor/Raça')
axes[0].set_ylim(-500, 6000) # Foco na área de maior densidade
axes[0].legend(title='Cor/Raça', bbox_to_anchor=(1.05, 1), loc='upper left')

# Gráfico 2: Barras (com IC 95% automático do Seaborn)
sns.barplot(data=df_censo, x='GRUPO', y='RENDA_TOTAL', hue='COR_NOME', ax=axes[1])
axes[1].set_title('Barras: Renda Média por Cor/Raça (com IC 95%)')
axes[1].legend(title='Cor/Raça', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

Qual a melhor visualização e porquê?
Para esta variável específica, o gráfico de barras revela-se a melhor visualização. Como a distribuição de renda no Brasil possui uma cauda direita extremamente longa (muita desigualdade e outliers), o gráfico de violino acaba por ficar visualmente "esmagado" perto do zero, dificultando a comparação entre as categorias. O gráfico de barras, por agregar os dados na média e exibir diretamente as barras de erro (intervalo de confiança a 95%), permite uma comparação de grandezas muito mais clara e imediata.

8.1 Os gráficos demonstram uma disparidade racial clara nos rendimentos. Tanto em Macapá quanto no resto do país, as populações declaradas branca e amarela apresentam médias de renda total significativamente superiores às das populações preta, parda e indígena. Em Macapá, o padrão de desigualdade reflete a tendência nacional, consolidando a cor/raça como um fator estrutural fortemente correlacionado com a remuneração.

In [ ]:
# (0,35) 9-Trace a renda média por faixa etária (crie faixas de 10 em 10 anos...
# usando um gráfico de linhas com intervalo de confiança (95%).

# 1. Criar faixas etárias de 10 em 10 anos
# Começamos em 10 porque removemos os menores de 10 anos na etapa de Qualidade de Dados
bins_idade = range(10, 130, 10)
labels_idade = [f"{i} a {i+9}" for i in range(10, 120, 10)]

df_censo['FAIXA_ETARIA_10_ANOS'] = pd.cut(
    df_censo['IDADE'],
    bins=bins_idade,
    labels=labels_idade,
    right=False
)

# 2. Gerar o gráfico de linhas (o Seaborn calcula o IC 95% automaticamente como uma sombra)
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df_censo,
    x='FAIXA_ETARIA_10_ANOS',
    y='RENDA_TOTAL',
    hue='GRUPO',
    marker='o'
)

plt.title('Ciclo de Vida da Renda: Renda Média por Faixa Etária (com IC 95%)')
plt.xlabel('Faixa Etária (anos)')
plt.ylabel('Renda Média Total (R$)')
plt.xticks(rotation=45)
plt.legend(title='Região')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

9.1 O gráfico ilustra claramente o "ciclo de vida" financeiro: a renda média cresce de forma acentuada a partir da entrada no mercado de trabalho, atinge o seu pico nas faixas dos 40 aos 59 anos (fase de consolidação da carreira) e começa a declinar com a aproximação da idade de reforma. No caso específico de Macapá, nota-se que nas faixas etárias mais avançadas (a partir dos 70 anos), a banda sombreada do intervalo de confiança alarga-se drasticamente. Isto indica que o tamanho da amostra local para idosos é pequeno demais para sustentar certezas estatísticas; quando os intervalos de confiança de duas faixas etárias vizinhas se sobrepõem, significa que a diferença de renda entre elas nesse trecho não é estatisticamente significativa.

In [ ]:
# (0,35) 10-Qual a relação entre o nível de instrução (NIVEL_INSTRUCAO) e a renda...
# Mostre uma tabela dinâmica e calcule a correlação de Spearman.

# 1. Tabela Dinâmica (Pivot Table)
pivot_instrucao = pd.pivot_table(
    df_censo,
    values='RENDA_TOTAL',
    index='NIVEL_INSTRUCAO',
    columns='GRUPO',
    aggfunc='mean'
)

print("--- Renda Média por Nível de Instrução e Grupo ---")
display(pivot_instrucao)

# 2. Correlação de Spearman
# Para o teste estatístico, precisamos remover "Não informado" (criado na Q6)
# e a categoria 5 ("Não determinado" segundo o dicionário), pois não são níveis ordinais.
df_valido = df_censo[~df_censo['NIVEL_INSTRUCAO'].isin(['Não informado', '5', '5.0', 5, 5.0])].copy()
df_valido['NIVEL_INSTRUCAO'] = pd.to_numeric(df_valido['NIVEL_INSTRUCAO'])

macapa_valido = df_valido[df_valido['GRUPO'] == MUNICIPIO]
resto_valido = df_valido[df_valido['GRUPO'] == 'Resto do pais']

corr_m, pval_m = stats.spearmanr(macapa_valido['NIVEL_INSTRUCAO'], macapa_valido['RENDA_TOTAL'])
corr_r, pval_r = stats.spearmanr(resto_valido['NIVEL_INSTRUCAO'], resto_valido['RENDA_TOTAL'])

print("\n--- Correlação de Spearman (Nível de Instrução vs Renda Total) ---")
print(f"{MUNICIPIO}: Coeficiente = {corr_m:.3f} | p-valor = {pval_m:.2e}")
print(f"Resto do país: Coeficiente = {corr_r:.3f} | p-valor = {pval_r:.2e}")

10.1 A tabela dinâmica e os testes de correlação de Spearman evidenciam uma relação positiva e estatisticamente significativa (p-valores próximos de zero) entre o nível de instrução e a renda. Tanto em Macapá quanto no resto do país, regista-se um coeficiente positivo, indicando que à medida que o nível de escolaridade de um indivíduo avança, a sua remuneração média cresce substancialmente. Estes dados confirmam o nível de instrução como um dos preditores mais fortes do poder de compra e da mobilidade social.

In [ ]:
# (0,3) 11-Qual a renda média de acordo com a posição na ocupação (POSICAO_OCUPACAO),
# tanto no seu município quanto no resto do país?

# 1. Mapeamento segundo o Dicionário de Dados do IBGE
dic_ocupacao = {
    '1': '1-Com carteira', '1.0': '1-Com carteira',
    '2': '2-Militares/Estatutários', '2.0': '2-Militares/Estatutários',
    '3': '3-Sem carteira', '3.0': '3-Sem carteira',
    '4': '4-Conta própria', '4.0': '4-Conta própria',
    '5': '5-Empregadores', '5.0': '5-Empregadores',
    '6': '6-Não remunerados', '6.0': '6-Não remunerados',
    '7': '7-Produção próprio consumo', '7.0': '7-Produção próprio consumo'
}

df_censo['POSICAO_NOME'] = df_censo['POSICAO_OCUPACAO'].astype(str).replace(dic_ocupacao)

# Filtrar apenas as linhas de quem tem uma ocupação válida preenchida
df_ocupados = df_censo[df_censo['POSICAO_NOME'].isin(dic_ocupacao.values())]

# 2. Criar a Tabela Dinâmica
pivot_ocupacao = pd.pivot_table(
    df_ocupados,
    values='RENDA_TOTAL',
    index='POSICAO_NOME',
    columns='GRUPO',
    aggfunc='mean'
)

print("--- Renda Média por Posição na Ocupação ---")
display(pivot_ocupacao)

11.1 Os resultados demonstram que os 'Empregadores' e os 'Militares e funcionários públicos estatutários' constituem as categorias com os maiores rendimentos médios, tanto a nível nacional como em Macapá. Em contrapartida, os trabalhadores 'Sem carteira assinada' e os trabalhadores por 'Conta própria' apresentam rendimentos substancialmente mais baixos. Estes dados refletem a realidade do mercado de trabalho brasileiro, evidenciando o elevado prémio salarial associado à formalidade e à estabilidade do setor público, em contraste com a vulnerabilidade financeira do trabalho informal.

In [ ]:
# (0,35) 12-Combine cor/raça e sexo em uma única tabela dinâmica de renda média,
# considerando apenas o seu município. O que se observa sobre desigualdade "interseccional"?

# Filtrar apenas os dados de Macapá e remover os "Não informados" para uma tabela limpa
df_macapa_limpo = df_censo[
    (df_censo['GRUPO'] == MUNICIPIO) &
    (df_censo['COR_NOME'] != 'Não informado') &
    (df_censo['SEXO_NOME'].isin(['Masculino', 'Feminino']))
]

# Tabela dinâmica cruzando as duas variáveis
pivot_interseccional = pd.pivot_table(
    df_macapa_limpo,
    values='RENDA_TOTAL',
    index='COR_NOME',
    columns='SEXO_NOME',
    aggfunc='mean'
)

# Calcular a diferença percentual entre os extremos para destacar a desigualdade
pivot_interseccional = pivot_interseccional.sort_values(by='Masculino', ascending=False)
display(pivot_interseccional)

12.1 A tabela revela a profunda desigualdade interseccional, mas com particularidades específicas na amostra de Macapá. De forma consistente com o padrão estrutural, os homens brancos ocupam o topo absoluto da pirâmide de rendimentos com larga vantagem. No entanto, nota-se uma anomalia estatística nas populações indígena e amarela: as mulheres indígenas apresentam rendimentos médios superiores às mulheres brancas, e os homens amarelos ganham menos do que a média dos homens pretos e pardos na cidade. Estes desvios explicam-se, muito provavelmente, pela reduzida representatividade amostral destas minorias raciais no município, onde a presença de poucos indivíduos com rendimentos atípicos (outliers) é suficiente para distorcer drasticamente a média do grupo.

In [ ]:
# (0,25) 13-Examine a cardinalidade (número de valores distintos) de COD_OCUPACAO e
# COD_ATIVIDADE_ECON, comparando com a de outras variáveis categóricas da base.

colunas_categoricas = ['COD_OCUPACAO', 'COD_ATIVIDADE_ECON', 'POSICAO_OCUPACAO',
                       'SUBGRUPO_OCUPACAO', 'NIVEL_INSTRUCAO', 'COR_RACA']

print("--- Cardinalidade das Variáveis Categóricas ---")
for col in colunas_categoricas:
    print(f"{col}: {df_censo[col].nunique()} valores distintos")

13.1 A altíssima cardinalidade de COD_OCUPACAO e COD_ATIVIDADE_ECON (com centenas de valores distintos) inviabiliza o uso de técnicas tradicionais como o One-Hot Encoding. Aplicar o One-Hot Encoding a estas variáveis criaria centenas de novas colunas no conjunto de dados, resultando numa matriz esparsa gigantesca. Isso leva ao problema conhecido como "Maldição da Dimensionalidade", que degrada drasticamente a performance de modelos de Machine Learning, aumenta o risco de overfitting e consome recursos computacionais excessivos.
Como estratégia alternativa para o TP2, em vez do One-Hot, poderíamos utilizar Target Encoding (substituir a categoria pela média da variável-alvo, a Renda, para aquela ocupação específica), ou realizar um agrupamento semântico (binning), agrupando os códigos finos em macrorregiões profissionais maiores baseadas nas diretrizes do IBGE.

In [ ]:
# (0,3) 14-A partir de IDADE, crie a coluna FAIXA_ETARIA com os seguintes intervalos...
bins_idade = [10, 15, 18, 25, 35, 45, 55, 65, 121]
labels_idade = ['[10..15)', '[15..18)', '[18..25)', '[25..35)', '[35..45)', '[45..55)', '[55..65)', '[65..120]']

# right=False garante o formato [a, b) onde inclui o 1º número e exclui o 2º
df_censo['FAIXA_ETARIA'] = pd.cut(df_censo['IDADE'], bins=bins_idade, labels=labels_idade, right=False)


# (0,25) 15-A partir de RENDA_TOTAL_SM, crie a coluna FAIXA_RENDA usando os cortes oficiais...
bins_renda = [-np.inf, 0.0, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 15.0, 20.0, np.inf]
labels_renda = [
    'Sem rendimento', 'Até 1/4 SM', 'Mais de 1/4 até 1/2 SM', 'Mais de 1/2 até 1 SM',
    'Mais de 1 até 2 SM', 'Mais de 2 até 3 SM', 'Mais de 3 até 5 SM', 'Mais de 5 até 10 SM',
    'Mais de 10 até 15 SM', 'Mais de 15 até 20 SM', 'Mais de 20 SM'
]

# right=True garante que o limite superior está incluído (ex: 'Até 1/4 SM' inclui o exato valor 0.25)
df_censo['FAIXA_RENDA'] = pd.cut(df_censo['RENDA_TOTAL_SM'], bins=bins_renda, labels=labels_renda, right=True)

# Exibe a distribuição em percentagem para respondermos à Questão 15.1
distribuicao_renda = df_censo['FAIXA_RENDA'].value_counts(normalize=True) * 100
print("--- Distribuição Percentual das Classes (FAIXA_RENDA) ---")
print(distribuicao_renda)

15.1 Não, a variável está fortemente desbalanceada. Uma grande percentagem da amostra concentra-se nas categorias de menor rendimento (ex.: "Sem rendimento" e até 2 SM), enquanto as faixas de topo ("Mais de 15 até 20 SM" e "Mais de 20 SM") representam frações residuais da população. Isto constitui um grave problema para modelos de classificação no TP2, pois algoritmos treinados nestes dados tendem a ficar enviesados, prevendo quase sempre a classe maioritária e ignorando as minoritárias, alcançando falsas taxas de precisão elevadas. Para mitigar o problema, no TP2 poderemos aplicar técnicas de reamostragem, como o Undersampling (reduzir as classes maiores) ou o Oversampling / SMOTE (gerar dados sintéticos para as classes menores), ou ainda atribuir pesos distintos às classes (class weights) durante o treino do algoritmo.

In [ ]:
# (0,7) 16 Que atributos são os mais importantes para prever FAIXA_RENDA...

from sklearn.feature_selection import f_classif, chi2, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

# 1. Separar apenas a cidade de interesse (Macapá)
df_q16 = df_censo[df_censo['GRUPO'] == MUNICIPIO].copy()

# 2. Selecionar candidatos (NUNCA incluir variáveis vazadas do alvo como RENDA_TOTAL)
features = ['IDADE', 'SEXO', 'COR_RACA', 'NIVEL_INSTRUCAO', 'POSICAO_OCUPACAO', 'NUM_MORADORES_DOMICILIO']

# Limpeza e codificação para o Scikit-Learn (Algoritmos não aceitam strings nem nulos)
X = df_q16[features].copy()
for col in features:
    X[col] = X[col].fillna('Nulo')
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# O Chi-quadrado (chi2) exige valores estritamente não-negativos
X = X - X.min()

# Preparar Alvo
y = df_q16['FAIXA_RENDA']
mascara_validos = y.notnull()
X = X[mascara_validos]
y_encoded = LabelEncoder().fit_transform(y[mascara_validos])

# 3. Calcular as Métricas
df_metricas = pd.DataFrame(index=features)

# a) Correlação Absoluta (Spearman)
df_metricas['Correlacao'] = [abs(stats.spearmanr(X[col], y_encoded)[0]) for col in features]

# b) ANOVA
df_metricas['ANOVA'], _ = f_classif(X, y_encoded)

# c) Qui-quadrado
df_metricas['Qui_Quadrado'], _ = chi2(X, y_encoded)

# d) Informação Mútua
df_metricas['Informacao_Mutua'] = mutual_info_classif(X, y_encoded, random_state=42)

# e) Random Forest (Pode demorar uns 10 a 20 segundos)
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1)
rf.fit(X, y_encoded)
df_metricas['Random_Forest'] = rf.feature_importances_

# 4. Normalização (0 a 1) e Agregação
scaler = MinMaxScaler()
df_norm = pd.DataFrame(scaler.fit_transform(df_metricas), columns=df_metricas.columns, index=df_metricas.index)
df_norm['Media_Agregada'] = df_norm.mean(axis=1)

# 5. Ordenação
ranking = df_norm.sort_values(by='Media_Agregada', ascending=False)

print(f"--- Ranking de Importância dos Atributos ({MUNICIPIO}) ---")
display(ranking)

16.1 As métricas mostram um forte consenso no topo e na base do ranking. A POSICAO_OCUPACAO foi a vencedora indiscutível, dominando quatro das cinco métricas (Correlação, ANOVA, Informação Mútua e Random Forest), o que faz sentido prático, visto que o vínculo laboral e a posição no mercado ditam a remuneração direta. A IDADE consolidou o segundo lugar, liderando no Qui-Quadrado, confirmando o peso do fator "ciclo de vida". A grande surpresa (para baixo) foi a posição fraca do NIVEL_INSTRUCAO, e a total irrelevância preditiva do SEXO e da COR_RACA nestes testes agregados univariados. Isto muda radicalmente a decisão para o TP2: para que o modelo tenha uma boa performance, terá de basear-se fortemente na engenharia dos atributos ocupacionais e etários, sendo arriscado depender apenas da escolaridade como preditor principal.